# Callas — build the dataset one clip at a time

Feed clips one by one (YouTube link **or** a local audio file you downloaded). Each clip is transcribed, split into **who speaks to whom** (caller/scammer vs victim), PII-masked, and appended to a growing **dataset** on your Google Drive.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4).
2. **HF token** (for caller/victim separation): create one + accept terms for `pyannote/speaker-diarization-3.1` and `pyannote/segmentation-3.0`. Skip → transcript only, one speaker.
3. YouTube blocks Colab a lot — easiest is to **download audio yourself** and use the *local audio* cell.

Run setup once, then run a clip cell per clip. The dataset persists on Drive.

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q "transformers<5" "accelerate>=0.30" "pyannote.audio==3.1.1" pythainlp python-crfsuite webvtt-py pandas pyyaml tqdm
!pip install -q -U yt-dlp
%cd /content
![ -d callas-dataset ] || git clone https://github.com/JoyChanya/callas-dataset.git
%cd /content/callas-dataset
!git pull -q
import torch; print('cuda', torch.cuda.is_available())

In [ ]:
# persist output to Drive so the dataset survives + grows across sessions
from google.colab import drive
drive.mount('/content/drive')
import os
DATA = '/content/drive/MyDrive/callas-data'
os.makedirs(DATA, exist_ok=True)
!rm -rf data && ln -s "$DATA" data
print('data ->', os.path.realpath('data'))

In [ ]:
# Hugging Face token — needed to tell caller/scammer from victim. Enter to skip.
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("Paste HF token (or press Enter to skip): ")

## A) Local audio file  (recommended)
Download the clip's audio on your own computer (`yt-dlp -x --audio-format mp3 "URL"`) or any tool, then upload it here. Accepts mp3 / m4a / wav / mp4.

In [ ]:
from google.colab import files
up = files.upload()                      # choose an audio/video file
AUDIO = next(iter(up))
!python -u process_one.py --audio "{AUDIO}" --device cuda --model biodatlab/whisper-th-large-v3-combined

## B) Straight from a YouTube link
Needs `youtube.cookiefile` (run the cookies cell below first). Often still blocked on Colab — if so use option A.

In [ ]:
# one-time: upload a youtube cookies.txt (export from a logged-in browser)
from google.colab import files
up = files.upload()
name = next(iter(up)); import os; os.replace(name, 'cookies.txt')
!sed -i 's|cookiefile: ""|cookiefile: "cookies.txt"|' config.yaml
!grep cookiefile config.yaml

In [ ]:
URL = "https://www.youtube.com/watch?v=PASTE_ID_HERE"
!python -u process_one.py "{URL}" --device cuda --model biodatlab/whisper-th-large-v3-combined

## The dataset so far
Every clip appends rows to `data/out/dataset.csv` (on Drive). View + download it any time.

In [ ]:
import pandas as pd
df = pd.read_csv('data/out/dataset.csv')
print(len(df), 'turns from', df['clip'].nunique(), 'clips')
df.tail(20)
# from google.colab import files; files.download('data/out/dataset.csv')